In [ ]:
!pip install -U langchain-text-splitters
!pip install -U langchain-community langchain-text-splitters langchain
!pip install pypdf
!pip install -U langchain-huggingface
!pip install -qU langchain-chroma
!pip install -qU langchain
!pip install -qU langchain-groq
!pip install sentence-transformers
!pip install -q gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.4/331.4 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━

In [ ]:
from langchain_groq import ChatGroq
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
import shutil
import gradio as gr
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
import re
import gc
import shutil
from google.colab import drive
from google.colab import userdata

In [ ]:
# --- MONTAGE DRIVE ET CHEMINS ---
drive.mount('/content/drive')

PROJECT_PATH = "/content/drive/MyDrive/Smart-HR-Assistant"
NEW_DATA_PATH = os.path.join(PROJECT_PATH, "nouveau_data")
ACTUEL_DATA_PATH = os.path.join(PROJECT_PATH, "actuel_data")
CHROMA_PATH = os.path.join(PROJECT_PATH, "chroma_db")

# Création des dossiers si nécessaire
for path in [NEW_DATA_PATH, ACTUEL_DATA_PATH, CHROMA_PATH]:
    os.makedirs(path, exist_ok=True)

# --- CONFIGURATION API ---
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
llm_chat = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0.5
)

llm_multi_query = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Formuler la question
contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", (
    "Ton unique job est de reformuler la question de l'utilisateur pour qu'elle soit "
    "compréhensible toute seule, en utilisant l'historique. "
    "REGLÈS : "
    "1. NE RÉPONDS PAS à la question. "
    "2. NE DONNE AUCUNE INFORMATION sur l'entreprise. "
    "3. Renvoie UNIQUEMENT la question reformulée. "
    "Exemple : Si l'historique parle de Mutuelle et que l'utilisateur dit 'C'est combien ?', "
    "tu réponds juste : 'Quel est le prix de la mutuelle ?'"
    )),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])
contextualize_q_chain = contextualize_q_prompt | llm_chat | StrOutputParser()

# Le rédacteur
instruction_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "Tu es un expert en analyse de documents RH. Ton rôle est de fournir des réponses "
        "précises et structurées à partir du contexte fourni ci-dessous.\n\n"
        "CONTEXTE :\n{context}\n\n"
        "DIRECTIVES DE PRÉCISION :\n"
        "1. SYNTHÈSE : Synthétise les points clés (noms, dates, montants).\n"
        "2. SOURCES : Pour chaque information importante, tu DOIS citer le document source. "
        "Exemple : 'La prime est de 300€ [Source: politique_primes.pdf]'.\n" # <--- LA CLÉ !
        "3. INTÉGRITÉ : Si l'info n'est pas dans le contexte, dis 'Je ne trouve pas cette info dans les documents fournis'.\n"
        "4. ANTI-MÉLANGE : Si deux documents se contredisent, précise-le (ex: 'Le doc A dit X, mais le doc B dit Y').\n"
        "5. FORMAT : Utilise des listes à puces et du gras pour les chiffres."
    )),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}"),
])

In [ ]:
def get_or_update_db(update=True):
    os.makedirs(os.path.dirname(CHROMA_PATH), exist_ok=True)
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

    # 1. Charger la base existante (ou en créer une vide)
    vector_db = Chroma(persist_directory=CHROMA_PATH, embedding_function=embeddings)

    # 2. Vérifier s'il y a du nouveau
    if update:
      new_files = [f for f in os.listdir(NEW_DATA_PATH) if f.endswith('.pdf')]

      if new_files:
          loader = PyPDFDirectoryLoader(NEW_DATA_PATH)
          new_docs = loader.load()

          text_splitter = RecursiveCharacterTextSplitter(
              chunk_size=1200,
              chunk_overlap=200,
              length_function=len,
              add_start_index=True
          )

          new_chunks = text_splitter.split_documents(new_docs)

          # Ajout à Chroma
          vector_db.add_documents(new_chunks)

          # Déplacement vers l'archive
          for file_name in new_files:
              shutil.move(os.path.join(NEW_DATA_PATH, file_name),
                          os.path.join(ACTUEL_DATA_PATH, file_name))
              print(f"Archivé : {file_name}")
      else:
          print("Aucun nouveau fichier. Utilisation de la base existante.")

    return vector_db

In [ ]:
def vider_db():
    # Vider la collection
    try:
        vector_db = get_or_update_db(False)
        vector_db.delete_collection()
        print("Collection vidée.")

    except Exception as e:
        print(f"Erreur lors du vidage : {e}")

    # Supprimer la collection
    for nom in os.listdir(CHROMA_PATH):
        item_path = os.path.join(path, nom)
        if os.path.isdir(item_path):
            try:
                shutil.rmtree(item_path)
                print(f"Dossier de collection supprimé : {nom}")
            except Exception as e:
                print(f"Impossible de supprimer {nom} : {e}")

    # Supprimer les fichiers PDF dans actuel_data
    for nom in os.listdir(ACTUEL_DATA_PATH):
        chemin_complet = os.path.join(ACTUEL_DATA_PATH, nom)
        try:
            if os.path.isfile(chemin_complet):
                os.unlink(chemin_complet)
            elif os.path.isdir(chemin_complet):
                shutil.rmtree(chemin_complet)
            print("Actuel data vidé")
        except Exception as e:
            print(f"Impossible de supprimer {nom} : {e}")
    return "✅ La base de données a été vidée avec succès."

In [ ]:
"""# 1. On récupère les données actuelles
data = get_or_update_db(False).get()

# 2. On compte les IDs
nombre_de_morceaux = len(data['ids'])

print(f"🔍 Vérification : La base contient {nombre_de_morceaux} morceaux.")

if nombre_de_morceaux == 0:
    print("✨ Confirmation : La base est totalement vide !")
else:
    print("⚠️ Attention : Il reste encore des données.")"""

'# 1. On récupère les données actuelles\ndata = get_or_update_db(False).get()\n\n# 2. On compte les IDs\nnombre_de_morceaux = len(data[\'ids\'])\n\nprint(f"🔍 Vérification : La base contient {nombre_de_morceaux} morceaux.")\n\nif nombre_de_morceaux == 0:\n    print("✨ Confirmation : La base est totalement vide !")\nelse:\n    print("⚠️ Attention : Il reste encore des données.")'

In [ ]:
# Exemple de recherche avec sources
"""vector_db = get_or_update_db()
query = "c'est quoi le regle interieur?"
docs = vector_db.similarity_search(query, k=3)

print("--- ANALYSE DES DOUBLONS ---")
for i, doc in enumerate(docs):
    # On affiche l'index de départ pour voir si c'est le même endroit
    index = doc.metadata.get('start_index', 'N/A')
    print(f"Morceau {i+1} | Source: {os.path.basename(doc.metadata['source'])}")
    print(f"Position dans le doc: {index}")
    print(f"Texte: {doc.page_content[:100].strip()}...")
    print("-" * 30)"""

'vector_db = get_or_update_db()\nquery = "c\'est quoi le regle interieur?"\ndocs = vector_db.similarity_search(query, k=3)\n\nprint("--- ANALYSE DES DOUBLONS ---")\nfor i, doc in enumerate(docs):\n    # On affiche l\'index de départ pour voir si c\'est le même endroit\n    index = doc.metadata.get(\'start_index\', \'N/A\')\n    print(f"Morceau {i+1} | Source: {os.path.basename(doc.metadata[\'source\'])}")\n    print(f"Position dans le doc: {index}")\n    print(f"Texte: {doc.page_content[:100].strip()}...")\n    print("-" * 30)'

In [ ]:
def respond(message, chat_history):
    global llm_chat, llm_multi_query, contextualize_q_chain

    # 1. Mise à jour de la base (ou récupération)
    vector_db = get_or_update_db()

    # --- NOUVEAU : TRANSFORME L'HISTORIQUE POUR LE LLM ---
    formatted_history = []
    for user_msg, ai_msg in chat_history:
        formatted_history.append(HumanMessage(content=user_msg))
        formatted_history.append(AIMessage(content=ai_msg))

    # --- ÉTAPE 2 : REFORMULATION (Le Secrétaire travaille ici) ---
    # Si c'est la 1ère question, on garde le message tel quel.
    # Sinon, on demande au LLM de clarifier la question par rapport à l'historique.
    if chat_history:
        standalone_question = contextualize_q_chain.invoke({
            "chat_history": formatted_history,
            "input": message
        })
        print(f"DEBUG - Question reformulée : {standalone_question} okk")
    else:
        standalone_question = message

    # --- ÉTAPE 3 : RECHERCHE AVANCÉE (Multi-Query) ---
    # On crée le retriever intelligent qui va générer des questions synonymes
    advanced_retriever = MultiQueryRetriever.from_llm(
        retriever=vector_db.as_retriever(search_kwargs={"k": 3}),
        llm=llm_multi_query
    )

    docs = advanced_retriever.invoke(input=standalone_question)
    context = "\n\n".join([d.page_content for d in docs])

    # --- ÉTAPE 4 : RÉPONSE FINALE (Le Rédacteur) ---
    chain = instruction_prompt | llm_chat
    result = chain.invoke({
        "context": context,
        "question": message, # On garde la question originale pour l'affichage
        "chat_history": formatted_history
    })

    # 5. Ajouter l'échange à l'historique
    chat_history.append((message, result.content))
    return "", chat_history

In [ ]:
respond("c'est quoi la regle ?", [])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Aucun nouveau fichier. Utilisation de la base existante.


('',
 [("c'est quoi la regle ?",
   "Je suis un expert en analyse de documents RH et je suis là pour vous aider. Selon les directives fournies, voici la règle :\n\n**1. SYNTHÈSE** : Je vais synthétiser les points clés (noms, dates, montants) des documents fournis.\n\n**2. SOURCES** : Pour chaque information importante, je vais citer le document source.\n\n**3. INTÉGRITÉ** : Si une information n'est pas dans le contexte, je vais le mentionner.\n\n**4. ANTI-MÉLANGE** : Si deux documents se contredisent, je vais préciser le conflit.\n\n**5. FORMAT** : Je vais utiliser des listes à puces et du gras pour les chiffres.\n\nC'est tout ! Je suis prêt à analyser les documents et à fournir les réponses attendues.")])

In [ ]:
def upload_files(files):
    # 1. Sécurité anti-vide
    if not files:
        gr.Warning("⚠️ Veuillez sélectionner au moins un fichier.")
        return "ERREUR : Aucun fichier sélectionné."

    # On s'assure que les dossiers existent
    os.makedirs(NEW_DATA_PATH, exist_ok=True)
    os.makedirs(ACTUEL_DATA_PATH, exist_ok=True)

    # On récupère les noms de fichiers déjà archivés
    actuel_files = set(os.listdir(ACTUEL_DATA_PATH))

    files_to_process = []
    doublons_count = 0

    for file in files:
        # On extrait le nom propre du fichier (ex: "charte.pdf")
        nom_fichier = os.path.basename(file.name)

        # COMPARAISON : On vérifie si le NOM est déjà dans le SET
        if nom_fichier not in actuel_files:
            destination = os.path.join(NEW_DATA_PATH, nom_fichier)
            # On copie le fichier temporaire vers ton dossier NEW_DATA
            shutil.copy(file.name, destination)
            files_to_process.append(nom_fichier)
        else:
            doublons_count += 1

    # 2. On ne lance l'IA que s'il y a du nouveau
    if files_to_process:
        get_or_update_db(update=True)
        msg = f"✅ SUCCÈS : {len(files_to_process)} fichiers ajoutés."
        if doublons_count > 0:
            msg += f" ({doublons_count} doublons ignorés)"
        return msg
    else:
        return "ℹ️ Tous ces fichiers sont déjà dans la base. Aucun nouveau fichier à synchroniser."

In [ ]:
vider_db()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Collection vidée.
Dossier de collection supprimé : 539bcff2-557e-41b0-b925-7def75904093
Actuel data vidé
Actuel data vidé
Actuel data vidé


'✅ La base de données a été vidée avec succès.'

In [29]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🤖 Assistant RH Intelligent")
    gr.Markdown("L'IA qui connaît vos documents internes sur le bout des doigts.")

    with gr.Tabs():
        # --- ONGLET 1 : CHAT ---
        with gr.Tab("💬 Chat avec l'Expert"):
            chatbot = gr.Chatbot(label="Discussion", height=500)
            with gr.Row():
                msg = gr.Textbox(
                    placeholder="Posez votre question (ex: Quelles sont les règles de mutuelle ?)",
                    show_label=False,
                    scale=9
                )
                submit_btn = gr.Button("Envoyer", variant="primary", scale=1)

            clear_chat = gr.ClearButton([msg, chatbot], value="Effacer la conversation")

        # --- ONGLET 2 : ADMINISTRATION ---
        with gr.Tab("⚙️ Administration"):
            gr.Markdown("### 📂 Gestion des documents")
            gr.Markdown("Ajoutez de nouveaux PDF pour mettre à jour la base de connaissances.")

            with gr.Row():
                upload_btn = gr.File(
                    label="Déposez vos PDF ici",
                    file_count="multiple",
                    file_types=[".pdf"]
                )

            # Bouton de validation principal
            process_btn = gr.Button("📥 Ajouter à la base de données", variant="primary")

            # Afficheur de statut
            status_label = gr.Textbox(
                              label="Statut du traitement",
                              placeholder="Les étapes de l'indexation s'afficheront ici...",
                              lines=10,
                              max_lines=15,
                              interactive=False,
                              show_copy_button=True
                          )
            gr.HTML("<br><br><br>") # Un peu d'espace avant la zone de danger
            gr.Markdown("---")

            # Zone de danger en bas à gauche
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("🔒 **Maintenance**")
                    clear_db_btn = gr.Button("🗑️ Vider la DB", variant="stop", size="sm")
                with gr.Column(scale=4):
                    # Cette colonne vide sert à pousser le bouton vers la gauche
                    pass

    # --- LOGIQUE DES BOUTONS ---

    # Envoi du message
    msg.submit(respond, [msg, chatbot], [msg, chatbot])
    submit_btn.click(respond, [msg, chatbot], [msg, chatbot])

    # Indexation des fichiers
    process_btn.click(
        fn=upload_files,
        inputs=upload_btn,
        outputs=status_label,
        show_progress="full"
    )

    # Vidage de la base
    clear_db_btn.click(
        fn=vider_db,
        outputs=status_label,
        show_progress="full"
    )

# Lancement de l'application
if __name__ == "__main__":
    demo.launch(debug=True)

/tmp/ipykernel_26914/2609561450.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_26914/2609561450.py:8: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="Discussion", height=500)
/tmp/ipykernel_26914/2609561450.py:8: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="Discussion", height=500)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c5833f5c7b7ae68c36.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://c5833f5c7b7ae68c36.gradio.live


In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("# 📁 Assistant Smart-HR - Espace Administration")

    with gr.Row():
        # Partie Upload pour le RH
        with gr.Column(scale=1):
            file_output = gr.File(file_count="multiple", label="Téléverser des PDF RH")
            upload_button = gr.Button("1. Stocker les fichiers")
            status_label = gr.Textbox(label="Statut du stockage")

            update_db_button = gr.Button("2. Synchroniser la base (IA)", variant="primary")
            db_status = gr.Textbox(label="Statut de l'IA")

        # Partie Chat pour l'utilisateur
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(label="Discussion avec vos documents")
            msg = gr.Textbox(label="Posez votre question RH")
            clear = gr.Button("Effacer l'historique")

    # --- LOGIQUE DES BOUTONS ---

    # Bouton 1 : On stocke les fichiers dans le Drive
    upload_button.click(upload_files, inputs=file_output, outputs=status_label, show_progress="hidden", queue=False)

    # Bouton 2 : On lance la fonction maj_base_de_donnees() qu'on a fait juste avant
    update_db_button.click(get_or_update_db, outputs=db_status)

    # Le reste de ta logique de chat (respond...)
    msg.submit(respond, [msg, chatbot], [msg, chatbot])
    clear.click(lambda: None, None, chatbot, queue=False)

demo.launch(debug=True)

/tmp/ipykernel_483/3760806977.py:16: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="Discussion avec vos documents")
/tmp/ipykernel_483/3760806977.py:16: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="Discussion avec vos documents")


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9b740a6a7f48fd22a2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 416, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1160, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🗑️ Doublon supprimé (déjà présent dans l'archive) : Fiche de poste RRH.pdf
Aucun nouveau fichier. Utilisation de la base existante.


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 416, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1160, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🗑️ Doublon supprimé (déjà présent dans l'archive) : charte_du_teletravail.pdf
🗑️ Doublon supprimé (déjà présent dans l'archive) : Fiche de poste RRH.pdf
Archivé : Rapport-Remuneration-2022.pdf
Archivé : ReglementInterieurModele.pdf
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7864 <> https://9b740a6a7f48fd22a2.gradio.live
